# Семинар: Прогнозирование временных рядов с LSTM
Данные: yfinance | Архитектура: PyTorch LSTM


## 1. Установка и импорт библиотек

In [ ]:
!pip install yfinance torch pandas numpy scikit-learn matplotlib

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

## 2. Загрузка данных

In [ ]:
df = yf.download('AAPL', start='2020-01-01', end='2025-06-01')
df = df[['Close']]
df.head()

## 3. Предобработка

In [ ]:
# Заполнение пропусков
df = df.interpolate()

# Масштабирование
scaler = MinMaxScaler()
data = scaler.fit_transform(df.values)

## 4. Формирование последовательностей

In [ ]:
def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

SEQ_LEN = 20
X, y = create_sequences(data, SEQ_LEN)
print(X.shape, y.shape)

## 5. Разбиение на train/test

In [ ]:
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

## 6. Dataset и DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_ds = TimeSeriesDataset(X_train, y_train)
test_ds = TimeSeriesDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

## 7. Определение модели LSTM

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = LSTMModel(input_dim=1, hidden_dim=50, num_layers=2)
print(model)

## 8. Обучение модели

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 30
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(Xb)
        loss = criterion(pred.view(-1), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss/len(train_loader):.6f}")

## 9. Оценка на тесте

In [ ]:
model.eval()
preds, actuals = [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        pred = model(Xb)
        preds.extend(pred.view(-1).numpy())
        actuals.extend(yb.numpy())

from sklearn.metrics import mean_squared_error, mean_absolute_error
mse = mean_squared_error(actuals, preds)
rmse = np.sqrt(mse)
mae = mean_absolute_error(actuals, preds)
print(f"Test MSE: {mse:.6f}, RMSE: {rmse:.6f}, MAE: {mae:.6f}")

## 10. Визуализация результатов

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(actuals, label='Actual')
plt.plot(preds, label='Predicted')
plt.legend()
plt.title('Actual vs Predicted Prices')
plt.show()